In [27]:

#chatbot using RNN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Dataset
questions = [
    "hi", "hello",
    "how are you",
    "what is ai",
    "bye"
]

labels = [
    "greeting", "greeting",
    "status",
    "ai",
    "bye"
]

responses = {
    "greeting": "Hello!",
    "status": "I am fine, how can I help you?",
    "ai": "AI stands for Artificial Intelligence.",
    "bye": "Goodbye!"
}

# Train model
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)

model = LogisticRegression()
model.fit(X, labels)

# Chat function
def chat(msg):
    intent = model.predict(vectorizer.transform([msg]))[0]
    return responses[intent]

# Chat loop
print("Chatbot ready! (type 'exit' to stop)")
while True:
    user = input("You: ")
    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))

Chatbot ready! (type 'exit' to stop)
You: hi
Bot: Hello!
You: what is ai
Bot: AI stands for Artificial Intelligence.
You: exit
Bot: Goodbye!


In [26]:
#chatbot using lstn
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset
questions = ["hi", "hello", "how are you", "what is ai", "bye"]
answers = ["hello", "hi", "i am fine", "ai is artificial intelligence", "goodbye"]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(questions + answers)

q_seq = pad_sequences(tokenizer.texts_to_sequences(questions), maxlen=5)
a_seq = pad_sequences(tokenizer.texts_to_sequences(answers), maxlen=5)

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(500, 32, input_length=5),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(5, activation='softmax')  # 5 responses
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Labels (0–4 for answers)
labels = np.array([0, 1, 2, 3, 4])

model.fit(q_seq, labels, epochs=200, verbose=0)

# Chat function
def chat(msg):
    seq = pad_sequences(tokenizer.texts_to_sequences([msg]), maxlen=5)
    pred = model.predict(seq, verbose=0)
    index = np.argmax(pred)
    return answers[index]

# Chat loop
print("LSTM Chatbot Ready! (type 'exit' to stop)")
while True:
    user = input("You: ")
    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


LSTM Chatbot Ready! (type 'exit' to stop)
You: hi


Bot: hello
You: ai
Bot: hello
You: what is ai
Bot: ai is artificial intelligence
You: exit
Bot: Goodbye!


In [28]:
#chatbot using BERT
!pip install transformers

from transformers import pipeline

# Load BERT QA model
qa = pipeline("question-answering")

# Context (important!)
context = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine learning is a subset of AI.
Deep learning is a part of machine learning.
"""

print("BERT Chatbot Ready! (type 'exit' to stop)")

while True:
    question = input("You: ")
    if question.lower() == "exit":
        print("Bot: Goodbye!")
        break

    result = qa(question=question, context=context)
    print("Bot:", result['answer'])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT Chatbot Ready! (type 'exit' to stop)
You: what is ai
Bot: the simulation of human intelligence in machines
You: exit
Bot: Goodbye!


In [34]:
!pip install -q transformers sentencepiece

In [37]:
#chatbot using GPT
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Predefined correct answers (VERY IMPORTANT)
fixed_answers = {
    "what is ai": "Artificial Intelligence is the simulation of human intelligence in machines.",
    "what is python": "Python is a programming language used for software development, AI, and data science.",
    "what is java": "Java is a high-level programming language used for building applications.",
    "what is english": "English is a widely spoken international language."
}

def chat(user_input):
    user_input = user_input.lower()

    # Step 1: Check fixed answers
    if user_input in fixed_answers:
        return fixed_answers[user_input]

    # Step 2: Otherwise use model
    prompt = f"Answer clearly: {user_input}"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=60)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

print("Accurate Chatbot Ready! (type 'exit' to stop)")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    print("Bot:", chat(user))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Accurate Chatbot Ready! (type 'exit' to stop)
You: what is ai
Bot: Artificial Intelligence is the simulation of human intelligence in machines.
You: what is python
Bot: Python is a programming language used for software development, AI, and data science.
You: exit
Bot: Goodbye!


In [46]:
#chatbot using TRANSFORMERS
!pip install -q transformers torch sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

questions = [
    "hi",
    "how are you",
    "what is ai",
    "what is python",
    "bye"
]

answers = [
    "hello",
    "i am fine",
    "ai is artificial intelligence",
    "python is a programming language",
    "goodbye"
]

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()


for epoch in range(30):
    for q, a in zip(questions, answers):

        input_text = "question: " + q
        target_text = a

        inputs = tokenizer(input_text, return_tensors="pt")
        labels = tokenizer(target_text, return_tensors="pt").input_ids

        outputs = model(
            input_ids=inputs.input_ids,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

print("Training Completed!")

# Chat function
def chat(user_input):
    model.eval()

    input_text = "question: " + user_input
    inputs = tokenizer(input_text, return_tensors="pt")

    outputs = model.generate(inputs.input_ids, max_length=50)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Chat loop
print("Chatbot Ready! (type 'exit' to stop)")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    print("Bot:", chat(user))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Training Completed!
Chatbot Ready! (type 'exit' to stop)
You:  hi
Bot: hello
You: exit
Bot: Goodbye!
